In [0]:
%sql
CREATE WIDGET TEXT catalog DEFAULT "bootcamp";
CREATE WIDGET TEXT bronze_schema DEFAULT "urbanmart_bronze_panagiotakopoulos_026";
CREATE WIDGET TEXT silver_schema DEFAULT "urbanmart_silver_panagiotakopoulos_026";
CREATE WIDGET TEXT gold_schema DEFAULT "urbanmart_gold_panagiotakopoulos_026";

USE CATALOG IDENTIFIER(:catalog);
USE SCHEMA IDENTIFIER(:silver_schema);

In [0]:
%sql
-- 1. CUSTOMERS
CREATE TABLE IF NOT EXISTS customers (
    customer_id STRING NOT NULL COMMENT 'Customer identifier',
    first_name STRING COMMENT 'Standardized first name (Unknown if blank)',
    last_name STRING COMMENT 'Standardized last name (Unknown if blank)',
    email STRING COMMENT 'Lower-case email, NULL if blank or invalid',
    gender STRING COMMENT 'F, M or Unknown',
    birth_date DATE COMMENT 'Customer birth date, NULL if blank or invalid',
    city STRING COMMENT 'Customer city (Unknown if blank)',
    region STRING COMMENT 'Customer region (Unknown if blank)',
    segment STRING COMMENT 'Customer segment (Unknown if blank)',
    signup_date DATE COMMENT 'Customer signup date, NULL if blank or invalid',
    _source_file STRING COMMENT 'Source file path',
    _ingested_at TIMESTAMP COMMENT 'Original ingestion timestamp'
)
USING DELTA
COMMENT 'Silver cleaned customer data';

-- 2. CUSTOMER CHANGES
CREATE TABLE IF NOT EXISTS customer_changes (
    customer_id STRING NOT NULL COMMENT 'Customer identifier',
    new_city STRING COMMENT 'Updated city (NULL = city not changed)',
    new_region STRING COMMENT 'Updated region (NULL = region not changed)',
    new_segment STRING COMMENT 'Updated segment (NULL = segment not changed)',
    effective_date DATE NOT NULL COMMENT 'Date the change becomes effective',
    _source_file STRING COMMENT 'Source file path',
    _ingested_at TIMESTAMP COMMENT 'Original ingestion timestamp'
)
USING DELTA
COMMENT 'Silver validated customer change events';

-- 3. PRODUCTS
CREATE TABLE IF NOT EXISTS products (
    product_id STRING NOT NULL COMMENT 'Product identifier',
    product_name STRING NOT NULL COMMENT 'Product name (Unknown if blank)',
    category STRING NOT NULL COMMENT 'Product category (Unknown if blank)',
    subcategory STRING COMMENT 'Product subcategory (Unknown if blank)',
    brand STRING COMMENT 'Product brand (Unknown if blank)',
    unit_cost DECIMAL(12,2) NOT NULL  COMMENT 'Product unit cost',
    list_price DECIMAL(12,2) NOT NULL  COMMENT 'Product list price',
    is_active BOOLEAN COMMENT 'Whether product is active, NULL if blank or invalid',
    _source_file STRING COMMENT 'Source file path',
    _ingested_at TIMESTAMP COMMENT 'Original ingestion timestamp'
)
USING DELTA
COMMENT 'Silver cleaned product catalogue';

-- 4. STORES
CREATE TABLE IF NOT EXISTS stores (
    store_id STRING NOT NULL COMMENT 'Store identifier',
    store_name STRING COMMENT 'Store name (Unknown if blank)',
    store_type STRING COMMENT 'Store type (Unknown if blank)',
    city STRING COMMENT 'Store city (Unknown if blank)',
    region STRING COMMENT 'Store region (Unknown if blank)',
    open_date DATE COMMENT 'Store opening date, NULL if blank, invalid or in the future',
    size_sqm INT COMMENT 'Store size in square metres, NULL if blank, invalid or negative',
    _source_file STRING COMMENT 'Source file path',
    _ingested_at TIMESTAMP COMMENT 'Original ingestion timestamp'
)
USING DELTA
COMMENT 'Silver cleaned store data';

-- 5. ORDERS
CREATE TABLE IF NOT EXISTS orders (
    order_id STRING NOT NULL  COMMENT 'Order identifier',
    order_date DATE COMMENT 'Order date, NULL if blank, invalid or in the future (gold maps to -1)',
    customer_id STRING COMMENT 'Customer identifier as received (NULL if blank)',
    store_id STRING COMMENT 'Store identifier as received (NULL if blank)',
    payment_method STRING COMMENT 'Standardized payment method (Unknown if blank or not recognised)',
    order_status STRING NOT NULL COMMENT 'Standardized order status',
    _source_file STRING COMMENT 'Source file path',
    _ingested_at TIMESTAMP COMMENT 'Original ingestion timestamp'
)
USING DELTA
COMMENT 'Silver cleaned order headers';

-- 6. ORDER ITEMS
CREATE TABLE IF NOT EXISTS order_items (
    order_line_id STRING NOT NULL COMMENT 'Order line identifier',
    order_id STRING NOT NULL COMMENT 'Order identifier',
    product_id STRING NOT NULL COMMENT 'Product identifier',
    quantity INT NOT NULL COMMENT 'Quantity ordered (at least 1)',
    unit_price DECIMAL(12,2) NOT NULL COMMENT 'Unit selling price',
    discount_pct DECIMAL(5,4) NOT NULL COMMENT 'Discount as a fraction 0..1 (0 if blank)',
    _source_file STRING COMMENT 'Source file path',
    _ingested_at TIMESTAMP COMMENT 'Original ingestion timestamp'
)
USING DELTA
COMMENT 'Silver cleaned order line data';

-- 7. ETL REJECTS
CREATE TABLE IF NOT EXISTS etl_rejects (
    source_table STRING COMMENT 'Name of the source table',
    record_id STRING COMMENT 'Business key of the rejected record (NULL if the key itself is missing)',
    reject_code STRING COMMENT 'MISSING_KEY, CONFLICT_KEY, DUP_EXACT, INVALID_VALUE or ORPHAN_FK',
    reject_reason STRING COMMENT 'Reason the record was rejected',
    record_json STRING COMMENT 'Original bronze record as JSON',
    _source_file STRING COMMENT 'Original source file path',
    rejected_at TIMESTAMP COMMENT 'Timestamp when the record was rejected'
)
USING DELTA
COMMENT 'ETL rejected records. Rebuilt per source table on every run.';

In [0]:
%sql
CREATE OR REPLACE TABLE stg_customers AS
WITH norm AS (
    SELECT
        NULLIF(TRIM(b.customer_id), '') AS customer_id,
        COALESCE(INITCAP(NULLIF(TRIM(b.first_name), '')), 'Unknown') AS first_name,
        COALESCE(INITCAP(NULLIF(TRIM(b.last_name), '')),  'Unknown') AS last_name,
        CASE
            WHEN LOWER(TRIM(b.email)) RLIKE '^[^@\\s]+@[^@\\s]+\\.[^@\\s]{2,}$'
            THEN LOWER(TRIM(b.email))
        END AS email,
        CASE UPPER(TRIM(b.gender))
            WHEN 'F' THEN 'F'
            WHEN 'FEMALE' THEN 'F'
            WHEN 'M' THEN 'M'
            WHEN 'MALE' THEN 'M'
            ELSE 'Unknown'
        END AS gender,
        TRY_TO_DATE(NULLIF(TRIM(b.birth_date), ''), 'yyyy-MM-dd') AS birth_date,
        COALESCE(INITCAP(NULLIF(TRIM(b.city), '')),    'Unknown') AS city,
        COALESCE(INITCAP(NULLIF(TRIM(b.region), '')),  'Unknown') AS region,
        COALESCE(INITCAP(NULLIF(TRIM(b.segment), '')), 'Unknown') AS segment,
        TRY_TO_DATE(NULLIF(TRIM(b.signup_date), ''), 'yyyy-MM-dd') AS signup_date,
        b._source_file,
        b._ingested_at,
        TO_JSON(STRUCT(
            b.customer_id, b.first_name, b.last_name, b.email, b.gender,
            b.birth_date, b.city, b.region, b.segment, b.signup_date, b._source_file
        )) AS record_json
    FROM IDENTIFIER(:bronze_schema || '.customers') AS b
),
keyed AS (
    SELECT
        n.*,
        TO_JSON(STRUCT(first_name, last_name, email, gender, birth_date,
                       city, region, segment, signup_date)) AS content_json
    FROM norm n
),
ranked AS (
    SELECT
        k.*,
        SIZE(COLLECT_SET(content_json) OVER (PARTITION BY customer_id)) AS n_versions,
        ROW_NUMBER() OVER (PARTITION BY customer_id, content_json ORDER BY record_json) AS copy_rn
    FROM keyed k
)
SELECT
    customer_id, first_name, last_name, email, gender, birth_date,
    city, region, segment, signup_date,
    _source_file, _ingested_at, record_json,
    CASE
        WHEN customer_id IS NULL THEN 'MISSING_KEY'
        WHEN birth_date > CURRENT_DATE() THEN 'INVALID_VALUE'
        WHEN n_versions > 1 THEN 'CONFLICT_KEY'
        WHEN copy_rn > 1 THEN 'DUP_EXACT'
    END AS reject_code,
    CASE
        WHEN customer_id IS NULL THEN 'Missing customer_id'
        WHEN birth_date > CURRENT_DATE() THEN 'Future birth_date'
        WHEN n_versions > 1 THEN 'Conflicting customer_id (different content for the same key)'
        WHEN copy_rn > 1 THEN 'Exact duplicate customer record'
    END AS reject_reason
FROM ranked;

INSERT OVERWRITE customers
SELECT customer_id, first_name, last_name, email, gender, birth_date,
       city, region, segment, signup_date, _source_file, _ingested_at
FROM stg_customers
WHERE reject_code IS NULL;

INSERT INTO etl_rejects
REPLACE WHERE source_table = 'customers'
SELECT 'customers', customer_id, reject_code, reject_reason, record_json, _source_file, CURRENT_TIMESTAMP()
FROM stg_customers
WHERE reject_code IS NOT NULL;

num_affected_rows,num_inserted_rows
8,8


In [0]:
%sql
CREATE OR REPLACE TABLE stg_customer_changes AS
WITH norm AS (
    SELECT
        NULLIF(TRIM(b.customer_id), '') AS customer_id,
        INITCAP(NULLIF(TRIM(b.new_city), '')) AS new_city,
        INITCAP(NULLIF(TRIM(b.new_region), '')) AS new_region,
        INITCAP(NULLIF(TRIM(b.new_segment), '')) AS new_segment,
        NULLIF(TRIM(b.effective_date), '') AS raw_effective_date,
        TRY_TO_DATE(NULLIF(TRIM(b.effective_date), ''), 'yyyy-MM-dd') AS effective_date,
        b._source_file,
        b._ingested_at,
        TO_JSON(STRUCT(
            b.customer_id, b.new_city, b.new_region, b.new_segment, b.effective_date, b._source_file
        )) AS record_json
    FROM IDENTIFIER(:bronze_schema || '.customer_changes') AS b
),
keyed AS (
    SELECT n.*, TO_JSON(STRUCT(new_city, new_region, new_segment)) AS content_json
    FROM norm n
),
ranked AS (
    SELECT
        k.*,
        SIZE(COLLECT_SET(content_json) OVER (PARTITION BY customer_id, effective_date)) AS n_versions,
        ROW_NUMBER() OVER (PARTITION BY customer_id, effective_date, content_json ORDER BY record_json) AS copy_rn
    FROM keyed k
)
SELECT
    r.customer_id, r.new_city, r.new_region, r.new_segment, r.effective_date,
    r._source_file, r._ingested_at, r.record_json,
    CASE
        WHEN r.customer_id IS NULL THEN 'MISSING_KEY'
        WHEN r.raw_effective_date IS NULL THEN 'MISSING_KEY'
        WHEN r.effective_date IS NULL THEN 'INVALID_VALUE'
        WHEN r.n_versions > 1 THEN 'CONFLICT_KEY'
        WHEN r.copy_rn > 1 THEN 'DUP_EXACT'
        WHEN c.customer_id IS NULL THEN 'ORPHAN_FK'
    END AS reject_code,
    CASE
        WHEN r.customer_id IS NULL THEN 'Missing customer_id'
        WHEN r.raw_effective_date IS NULL THEN 'Missing effective_date'
        WHEN r.effective_date IS NULL THEN 'Invalid effective_date'
        WHEN r.n_versions > 1 THEN 'Conflicting change for the same customer_id and effective_date'
        WHEN r.copy_rn > 1 THEN 'Exact duplicate customer change record'
        WHEN c.customer_id IS NULL THEN 'Unknown customer_id (customer missing or rejected in silver)'
    END AS reject_reason
FROM ranked r
LEFT JOIN IDENTIFIER(:silver_schema || '.customers') AS c
    ON c.customer_id = r.customer_id;

INSERT OVERWRITE customer_changes
SELECT customer_id, new_city, new_region, new_segment, effective_date, _source_file, _ingested_at
FROM stg_customer_changes
WHERE reject_code IS NULL;

INSERT INTO etl_rejects
REPLACE WHERE source_table = 'customer_changes'
SELECT 'customer_changes', customer_id, reject_code, reject_reason, record_json, _source_file, CURRENT_TIMESTAMP()
FROM stg_customer_changes
WHERE reject_code IS NOT NULL;

num_affected_rows,num_inserted_rows
0,0


In [0]:
%sql
CREATE OR REPLACE TABLE stg_products AS
WITH raw AS (
    SELECT
        NULLIF(TRIM(b.product_id), '') AS product_id,
        COALESCE(NULLIF(TRIM(b.product_name), ''), 'Unknown') AS product_name,
        COALESCE(INITCAP(NULLIF(TRIM(b.category), '')),    'Unknown') AS category,
        COALESCE(INITCAP(NULLIF(TRIM(b.subcategory), '')), 'Unknown') AS subcategory,
        COALESCE(INITCAP(NULLIF(TRIM(b.brand), '')),       'Unknown') AS brand,
        NULLIF(TRIM(b.unit_cost), '') AS raw_unit_cost,
        NULLIF(TRIM(b.list_price), '') AS raw_list_price,
        CASE UPPER(TRIM(b.is_active)) WHEN 'Y' THEN TRUE WHEN 'N' THEN FALSE END AS is_active,
        b._source_file,
        b._ingested_at,
        TO_JSON(STRUCT(
            b.product_id, b.product_name, b.category, b.subcategory, b.brand,
            b.unit_cost, b.list_price, b.is_active, b._source_file
        )) AS record_json
    FROM IDENTIFIER(:bronze_schema || '.products') AS b
),
parsed AS (
    SELECT
        r.*,
        TRY_CAST(raw_unit_cost  AS DECIMAL(12,2)) AS unit_cost,
        TRY_CAST(raw_list_price AS DECIMAL(12,2)) AS list_price,
        CASE
            WHEN raw_unit_cost IS NULL THEN 'Missing unit_cost'
            WHEN TRY_CAST(raw_unit_cost AS DECIMAL(12,2)) IS NULL THEN 'Invalid unit_cost'
            WHEN LENGTH(REGEXP_EXTRACT(raw_unit_cost, '\\.([0-9]+)$', 1)) > 2
                THEN 'unit_cost has more than two decimal places'
            WHEN TRY_CAST(raw_unit_cost AS DECIMAL(12,2)) < 0 THEN 'Negative unit_cost'
        END AS unit_cost_issue,
        CASE
            WHEN raw_list_price IS NULL THEN 'Missing list_price'
            WHEN TRY_CAST(raw_list_price AS DECIMAL(12,2)) IS NULL THEN 'Invalid list_price'
            WHEN LENGTH(REGEXP_EXTRACT(raw_list_price, '\\.([0-9]+)$', 1)) > 2
                THEN 'list_price has more than two decimal places'
            WHEN TRY_CAST(raw_list_price AS DECIMAL(12,2)) < 0 THEN 'Negative list_price'
        END AS list_price_issue
    FROM raw r
),
keyed AS (
    SELECT
        p.*,
        TO_JSON(STRUCT(product_name, category, subcategory, brand,
                       unit_cost, list_price, is_active)) AS content_json
    FROM parsed p
),
ranked AS (
    SELECT
        k.*,
        SIZE(COLLECT_SET(content_json) OVER (PARTITION BY product_id)) AS n_versions,
        ROW_NUMBER() OVER (PARTITION BY product_id, content_json ORDER BY record_json) AS copy_rn
    FROM keyed k
)
SELECT
    product_id, product_name, category, subcategory, brand, unit_cost, list_price, is_active,
    _source_file, _ingested_at, record_json,
    CASE
        WHEN product_id IS NULL THEN 'MISSING_KEY'
        WHEN unit_cost_issue IS NOT NULL THEN 'INVALID_VALUE'
        WHEN list_price_issue IS NOT NULL THEN 'INVALID_VALUE'
        WHEN n_versions > 1 THEN 'CONFLICT_KEY'
        WHEN copy_rn > 1 THEN 'DUP_EXACT'
    END AS reject_code,
    CASE
        WHEN product_id IS NULL THEN 'Missing product_id'
        WHEN unit_cost_issue IS NOT NULL THEN unit_cost_issue
        WHEN list_price_issue IS NOT NULL THEN list_price_issue
        WHEN n_versions > 1 THEN 'Conflicting product_id (different content for the same key)'
        WHEN copy_rn > 1 THEN 'Exact duplicate product record'
    END AS reject_reason
FROM ranked;

INSERT OVERWRITE products
SELECT product_id, product_name, category, subcategory, brand,
       unit_cost, list_price, is_active, _source_file, _ingested_at
FROM stg_products
WHERE reject_code IS NULL;

INSERT INTO etl_rejects
REPLACE WHERE source_table = 'products'
SELECT 'products', product_id, reject_code, reject_reason, record_json, _source_file, CURRENT_TIMESTAMP()
FROM stg_products
WHERE reject_code IS NOT NULL;

num_affected_rows,num_inserted_rows
0,0


In [0]:
%sql
CREATE OR REPLACE TABLE stg_stores AS
WITH norm AS (
    SELECT
        NULLIF(TRIM(b.store_id), '') AS store_id,
        COALESCE(NULLIF(TRIM(b.store_name), ''), 'Unknown') AS store_name,
        COALESCE(INITCAP(NULLIF(TRIM(b.store_type), '')), 'Unknown') AS store_type,
        COALESCE(INITCAP(NULLIF(TRIM(b.city), '')), 'Unknown') AS city,
        COALESCE(INITCAP(NULLIF(TRIM(b.region), '')), 'Unknown') AS region,
        CASE
            WHEN TRY_TO_DATE(NULLIF(TRIM(b.open_date), ''), 'yyyy-MM-dd') <= CURRENT_DATE()
            THEN TRY_TO_DATE(NULLIF(TRIM(b.open_date), ''), 'yyyy-MM-dd')
        END AS open_date,
        CASE
            WHEN TRY_CAST(NULLIF(TRIM(b.size_sqm), '') AS INT) >= 0
            THEN TRY_CAST(NULLIF(TRIM(b.size_sqm), '') AS INT)
        END AS size_sqm,
        b._source_file,
        b._ingested_at,
        TO_JSON(STRUCT(
            b.store_id, b.store_name, b.store_type, b.city, b.region,
            b.open_date, b.size_sqm, b._source_file
        )) AS record_json
    FROM IDENTIFIER(:bronze_schema || '.stores') AS b
),
keyed AS (
    SELECT
        n.*,
        TO_JSON(STRUCT(store_name, store_type, city, region, open_date, size_sqm)) AS content_json
    FROM norm n
),
ranked AS (
    SELECT
        k.*,
        SIZE(COLLECT_SET(content_json) OVER (PARTITION BY store_id)) AS n_versions,
        ROW_NUMBER() OVER (PARTITION BY store_id, content_json ORDER BY record_json) AS copy_rn
    FROM keyed k
)
SELECT
    store_id, store_name, store_type, city, region, open_date, size_sqm,
    _source_file, _ingested_at, record_json,
    CASE
        WHEN store_id IS NULL THEN 'MISSING_KEY'
        WHEN n_versions > 1 THEN 'CONFLICT_KEY'
        WHEN copy_rn > 1 THEN 'DUP_EXACT'
    END AS reject_code,
    CASE
        WHEN store_id IS NULL THEN 'Missing store_id'
        WHEN n_versions > 1 THEN 'Conflicting store_id (different content for the same key)'
        WHEN copy_rn > 1 THEN 'Exact duplicate store record'
    END AS reject_reason
FROM ranked;

INSERT OVERWRITE stores
SELECT store_id, store_name, store_type, city, region, open_date, size_sqm,
       _source_file, _ingested_at
FROM stg_stores
WHERE reject_code IS NULL;

INSERT INTO etl_rejects
REPLACE WHERE source_table = 'stores'
SELECT 'stores', store_id, reject_code, reject_reason, record_json, _source_file, CURRENT_TIMESTAMP()
FROM stg_stores
WHERE reject_code IS NOT NULL;

num_affected_rows,num_inserted_rows
0,0


In [0]:
%sql
CREATE OR REPLACE TABLE stg_orders AS
WITH raw AS (
    SELECT
        NULLIF(TRIM(b.order_id), '') AS order_id,
        NULLIF(TRIM(b.order_ts), '') AS raw_ts,
        NULLIF(TRIM(b.customer_id), '') AS customer_id,
        NULLIF(TRIM(b.store_id), '') AS store_id,
        INITCAP(NULLIF(TRIM(b.payment_method), '')) AS payment_raw,
        INITCAP(NULLIF(TRIM(b.order_status), '')) AS order_status,
        b._source_file,
        b._ingested_at,
        TO_JSON(STRUCT(
            b.order_id, b.order_ts, b.customer_id, b.store_id,
            b.payment_method, b.order_status, b._source_file
        )) AS record_json
    FROM IDENTIFIER(:bronze_schema || '.orders') AS b
),
parsed AS (
    SELECT
        r.*,
        CAST(COALESCE(
            try_to_timestamp(raw_ts, 'yyyy-MM-dd HH:mm:ss'),
            try_to_timestamp(raw_ts, 'yyyy-MM-dd HH:mm'),
            try_to_timestamp(raw_ts, 'yyyy-MM-dd'),
            try_to_timestamp(raw_ts, 'dd/MM/yyyy HH:mm'),
            try_to_timestamp(raw_ts, 'dd/MM/yyyy')
        ) AS DATE) AS parsed_date
    FROM raw r
),
norm AS (
    SELECT
        order_id,
        CASE WHEN parsed_date <= CURRENT_DATE() THEN parsed_date END AS order_date,
        customer_id,
        store_id,
        CASE WHEN payment_raw IN ('Mobile Wallet', 'Card', 'Cash') THEN payment_raw ELSE 'Unknown' END AS payment_method,
        order_status,
        _source_file,
        _ingested_at,
        record_json
    FROM parsed
),
keyed AS (
    SELECT
        n.*,
        TO_JSON(STRUCT(order_date, customer_id, store_id, payment_method, order_status)) AS content_json
    FROM norm n
),
ranked AS (
    SELECT
        k.*,
        SIZE(COLLECT_SET(content_json) OVER (PARTITION BY order_id)) AS n_versions,
        ROW_NUMBER() OVER (PARTITION BY order_id, content_json ORDER BY record_json) AS copy_rn
    FROM keyed k
)
SELECT
    order_id, order_date, customer_id, store_id, payment_method, order_status,
    _source_file, _ingested_at, record_json,
    CASE
        WHEN order_id IS NULL THEN 'MISSING_KEY'
        WHEN order_status IS NULL THEN 'INVALID_VALUE'
        WHEN order_status NOT IN ('Cancelled', 'Completed', 'Returned') THEN 'INVALID_VALUE'
        WHEN n_versions > 1 THEN 'CONFLICT_KEY'
        WHEN copy_rn > 1 THEN 'DUP_EXACT'
    END AS reject_code,
    CASE
        WHEN order_id IS NULL THEN 'Missing order_id'
        WHEN order_status IS NULL THEN 'Missing order_status'
        WHEN order_status NOT IN ('Cancelled', 'Completed', 'Returned') THEN 'Invalid order_status'
        WHEN n_versions > 1 THEN 'Conflicting order_id (different content for the same key)'
        WHEN copy_rn > 1 THEN 'Exact duplicate order record'
    END AS reject_reason
FROM ranked;

INSERT OVERWRITE orders
SELECT order_id, order_date, customer_id, store_id, payment_method, order_status,
       _source_file, _ingested_at
FROM stg_orders
WHERE reject_code IS NULL;

INSERT INTO etl_rejects
REPLACE WHERE source_table = 'orders'
SELECT 'orders', order_id, reject_code, reject_reason, record_json, _source_file, CURRENT_TIMESTAMP()
FROM stg_orders
WHERE reject_code IS NOT NULL;

num_affected_rows,num_inserted_rows
0,0


In [0]:
%sql
CREATE OR REPLACE TABLE stg_order_items AS
WITH raw AS (
    SELECT
        NULLIF(TRIM(b.order_line_id), '') AS order_line_id,
        NULLIF(TRIM(b.order_id), '') AS order_id,
        NULLIF(TRIM(b.product_id), '') AS product_id,
        NULLIF(TRIM(b.quantity), '') AS raw_quantity,
        NULLIF(TRIM(b.unit_price), '') AS raw_unit_price,
        NULLIF(TRIM(b.discount_pct), '') AS raw_discount,
        b._source_file,
        b._ingested_at,
        TO_JSON(STRUCT(
            b.order_line_id, b.order_id, b.product_id, b.quantity,
            b.unit_price, b.discount_pct, b._source_file
        )) AS record_json
    FROM IDENTIFIER(:bronze_schema || '.order_items') AS b
),
parsed AS (
    SELECT
        r.*,
        TRY_CAST(raw_quantity AS INT) AS quantity,
        TRY_CAST(raw_unit_price AS DECIMAL(12,2)) AS unit_price,
        TRY_CAST(raw_discount AS DOUBLE) AS discount_num
    FROM raw r
),
norm AS (
    SELECT
        p.*,
        CASE
            WHEN raw_quantity IS NULL THEN 'Missing quantity'
            WHEN quantity IS NULL THEN 'Invalid quantity'
            WHEN quantity < 1 THEN 'Quantity must be at least 1'
        END AS quantity_issue,
        CASE
            WHEN raw_unit_price IS NULL THEN 'Missing unit_price'
            WHEN unit_price IS NULL THEN 'Invalid unit_price'
            WHEN LENGTH(REGEXP_EXTRACT(raw_unit_price, '\\.([0-9]+)$', 1)) > 2
                THEN 'unit_price has more than two decimal places'
            WHEN unit_price < 0 THEN 'Negative unit_price'
        END AS price_issue,
        CASE
            WHEN raw_discount IS NOT NULL AND discount_num IS NULL THEN 'Invalid discount_pct'
            WHEN discount_num < 0 OR discount_num > 100 THEN 'discount_pct out of range'
        END AS discount_issue,
        CASE
            WHEN raw_discount IS NULL THEN CAST(0 AS DECIMAL(5,4))
            WHEN discount_num BETWEEN 0 AND 1 THEN CAST(discount_num AS DECIMAL(5,4))
            WHEN discount_num > 1 AND discount_num <= 100 THEN CAST(discount_num / 100 AS DECIMAL(5,4))
        END AS discount_pct
    FROM parsed p
),
keyed AS (
    SELECT
        n.*,
        TO_JSON(STRUCT(order_id, product_id, quantity, unit_price, discount_pct)) AS content_json
    FROM norm n
),
ranked AS (
    SELECT
        k.*,
        SIZE(COLLECT_SET(content_json) OVER (PARTITION BY order_line_id)) AS n_versions,
        ROW_NUMBER() OVER (PARTITION BY order_line_id, content_json ORDER BY record_json) AS copy_rn
    FROM keyed k
)
SELECT
    r.order_line_id, r.order_id, r.product_id, r.quantity, r.unit_price, r.discount_pct,
    r._source_file, r._ingested_at, r.record_json,
    CASE
        WHEN r.order_line_id IS NULL OR r.order_id IS NULL OR r.product_id IS NULL THEN 'MISSING_KEY'
        WHEN r.quantity_issue IS NOT NULL OR r.price_issue IS NOT NULL OR r.discount_issue IS NOT NULL THEN 'INVALID_VALUE'
        WHEN r.n_versions > 1 THEN 'CONFLICT_KEY'
        WHEN r.copy_rn > 1 THEN 'DUP_EXACT'
        WHEN o.order_id IS NULL THEN 'ORPHAN_FK'
        WHEN p.product_id IS NULL THEN 'ORPHAN_FK'
    END AS reject_code,
    CASE
        WHEN r.order_line_id IS NULL THEN 'Missing order_line_id'
        WHEN r.order_id IS NULL THEN 'Missing order_id'
        WHEN r.product_id IS NULL THEN 'Missing product_id'
        WHEN r.quantity_issue IS NOT NULL THEN r.quantity_issue
        WHEN r.price_issue IS NOT NULL THEN r.price_issue
        WHEN r.discount_issue IS NOT NULL THEN r.discount_issue
        WHEN r.n_versions > 1 THEN 'Conflicting order_line_id (different content for the same key)'
        WHEN r.copy_rn > 1 THEN 'Exact duplicate order item record'
        WHEN o.order_id IS NULL THEN 'Unknown order_id (order missing or rejected in silver)'
        WHEN p.product_id IS NULL THEN 'Unknown product_id (product missing or rejected in silver)'
    END AS reject_reason
FROM ranked r
LEFT JOIN IDENTIFIER(:silver_schema || '.orders')   AS o ON o.order_id   = r.order_id
LEFT JOIN IDENTIFIER(:silver_schema || '.products') AS p ON p.product_id = r.product_id;

INSERT OVERWRITE order_items
SELECT order_line_id, order_id, product_id, quantity, unit_price, discount_pct,
       _source_file, _ingested_at
FROM stg_order_items
WHERE reject_code IS NULL;

INSERT INTO etl_rejects
REPLACE WHERE source_table = 'order_items'
SELECT 'order_items', order_line_id, reject_code, reject_reason, record_json, _source_file, CURRENT_TIMESTAMP()
FROM stg_order_items
WHERE reject_code IS NOT NULL;

num_affected_rows,num_inserted_rows
115,115


In [0]:
%sql
WITH r AS (
    SELECT 'customers' AS t,
           (SELECT COUNT(*) FROM IDENTIFIER(:bronze_schema || '.customers')) AS bronze_rows,
           (SELECT COUNT(*) FROM customers) AS silver_rows,
           (SELECT COUNT(DISTINCT customer_id) FROM customers) AS silver_keys,
           (SELECT COUNT(*) FROM etl_rejects WHERE source_table = 'customers') AS reject_rows
    UNION ALL
    SELECT 'customer_changes',
           (SELECT COUNT(*) FROM IDENTIFIER(:bronze_schema || '.customer_changes')),
           (SELECT COUNT(*) FROM customer_changes),
           (SELECT COUNT(DISTINCT customer_id, effective_date) FROM customer_changes),
           (SELECT COUNT(*) FROM etl_rejects WHERE source_table = 'customer_changes')
    UNION ALL
    SELECT 'products',
           (SELECT COUNT(*) FROM IDENTIFIER(:bronze_schema || '.products')),
           (SELECT COUNT(*) FROM products),
           (SELECT COUNT(DISTINCT product_id) FROM products),
           (SELECT COUNT(*) FROM etl_rejects WHERE source_table = 'products')
    UNION ALL
    SELECT 'stores',
           (SELECT COUNT(*) FROM IDENTIFIER(:bronze_schema || '.stores')),
           (SELECT COUNT(*) FROM stores),
           (SELECT COUNT(DISTINCT store_id) FROM stores),
           (SELECT COUNT(*) FROM etl_rejects WHERE source_table = 'stores')
    UNION ALL
    SELECT 'orders',
           (SELECT COUNT(*) FROM IDENTIFIER(:bronze_schema || '.orders')),
           (SELECT COUNT(*) FROM orders),
           (SELECT COUNT(DISTINCT order_id) FROM orders),
           (SELECT COUNT(*) FROM etl_rejects WHERE source_table = 'orders')
    UNION ALL
    SELECT 'order_items',
           (SELECT COUNT(*) FROM IDENTIFIER(:bronze_schema || '.order_items')),
           (SELECT COUNT(*) FROM order_items),
           (SELECT COUNT(DISTINCT order_line_id) FROM order_items),
           (SELECT COUNT(*) FROM etl_rejects WHERE source_table = 'order_items')
)
SELECT
    t AS table_name, bronze_rows, silver_rows, reject_rows,
    assert_true(bronze_rows = silver_rows + reject_rows,
        format_string('%s: bronze rows (%d) <> silver rows (%d) + rejects (%d)',
                      t, bronze_rows, silver_rows, reject_rows)) AS reconciles,
    assert_true(silver_rows = silver_keys,
        format_string('%s: duplicate primary keys in silver', t)) AS unique_keys
FROM r
ORDER BY t;

-- Referential integrity inside silver, and every reject has a code
SELECT
    assert_true((SELECT COUNT(*) FROM order_items i LEFT ANTI JOIN orders o   ON i.order_id   = o.order_id)   = 0,
        'order_items has an order_id that is not in silver orders')   AS items_to_orders_ok,
    assert_true((SELECT COUNT(*) FROM order_items i LEFT ANTI JOIN products p ON i.product_id = p.product_id) = 0,
        'order_items has a product_id that is not in silver products') AS items_to_products_ok,
    assert_true((SELECT COUNT(*) FROM etl_rejects WHERE reject_code IS NULL) = 0,
        'etl_rejects has rows without a reject_code')                  AS rejects_have_codes;

items_to_orders_ok,items_to_products_ok,rejects_have_codes
null,null,null
